# 05 — Final Load & Tableau Prep
**MediScope · DVA Capstone Project**

**Goal:** Load the fully-cleaned master dataset (`clean_lab_master_v3.csv`), perform
final aggregations and joins, and export Tableau-ready CSV files to `data/processed/`.

## Pipeline Context
| Notebook | Role |
|---|---|
| 01_extraction | Load raw CSVs → Save to processed/ |
| 02_cleaning | Clean & engineer features |
| 03_eda | Exploratory analysis & visualisations |
| 04_statistical_analysis | Hypothesis tests & models |
| **05_final_load_prep** (this) | Merge → Aggregate → Export Tableau CSVs |

## 1. Imports & Path Configuration

In [ ]:
import os
import pandas as pd
import numpy as np

# ── Resolve paths relative to this notebook's location ──────────────────────
NOTEBOOK_DIR  = os.path.dirname(os.path.abspath('__file__'))  # notebooks/
ROOT_DIR      = os.path.join(NOTEBOOK_DIR, '..')
PROCESSED_DIR = os.path.join(ROOT_DIR, 'data', 'processed')
TABLEAU_DIR   = os.path.join(ROOT_DIR, 'data', 'processed', 'tableau_exports')

os.makedirs(TABLEAU_DIR, exist_ok=True)

print('PROCESSED_DIR :', os.path.abspath(PROCESSED_DIR))
print('TABLEAU_DIR   :', os.path.abspath(TABLEAU_DIR))

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

## 2. Load the Cleaned Master Dataset

`clean_lab_master_v3.csv` is produced by `02_cleaning.ipynb` and contains
**76,069 rows × 18 columns**, joining LABEVENTS with patient/admission metadata
and adding engineered features (`los_days`, `is_abnormal`, etc.).

In [ ]:
# Try the local processed/ path first; fall back to the Colab path used during development
master_local  = os.path.join(PROCESSED_DIR, 'clean_lab_master_v3.csv')
master_colab  = '/content/clean_lab_master_v3 (1).csv'  # used in development

if os.path.exists(master_local):
    master_path = master_local
elif os.path.exists(master_colab):
    master_path = master_colab
else:
    raise FileNotFoundError(
        'clean_lab_master_v3.csv not found.\n'
        'Please run 02_cleaning.ipynb first and copy the output to data/processed/.'
    )

lab = pd.read_csv(master_path, low_memory=False)
lab['charttime'] = pd.to_datetime(lab['charttime'], errors='coerce')

print(f'Loaded: {os.path.basename(master_path)}')
print(f'Shape : {lab.shape}')
print(f'Cols  : {lab.columns.tolist()}')

## 3. Quick Sanity Check

In [ ]:
print('Null values per column:')
print(lab.isnull().sum())
lab.head(3)

## 4. Tableau Export 1 — Patient-Level Summary

One row per patient with aggregated lab statistics.

In [ ]:
patient_summary = (
    lab.groupby(['subject_id', 'gender', 'admission_type'])
    .agg(
        total_tests    = ('itemid',     'count'),
        abnormal_tests = ('is_abnormal','sum'),
        avg_valuenum   = ('valuenum',   'mean'),
        avg_los_days   = ('los_days',   'mean'),
        unique_items   = ('itemid',     'nunique'),
    )
    .reset_index()
)

patient_summary['abnormal_rate'] = (
    patient_summary['abnormal_tests'] / patient_summary['total_tests']
).round(4)

print(f'Patient summary shape: {patient_summary.shape}')
patient_summary.head()

## 5. Tableau Export 2 — Lab Category Breakdown

Aggregated statistics per fluid/category type — useful for bar/pie charts.

In [ ]:
category_breakdown = (
    lab.groupby(['category', 'fluid'])
    .agg(
        test_count     = ('itemid',     'count'),
        abnormal_count = ('is_abnormal','sum'),
        mean_value     = ('valuenum',   'mean'),
        median_value   = ('valuenum',   'median'),
    )
    .reset_index()
)

category_breakdown['abnormal_rate'] = (
    category_breakdown['abnormal_count'] / category_breakdown['test_count']
).round(4)

print(f'Category breakdown shape: {category_breakdown.shape}')
category_breakdown.head(10)

## 6. Tableau Export 3 — Monthly Lab Trends

Time-series data for Tableau line/area charts.

In [ ]:
monthly_trends = (
    lab.set_index('charttime')
    .resample('ME')  # Month-End frequency
    .agg(
        test_count     = ('itemid',     'count'),
        abnormal_count = ('is_abnormal','sum'),
        avg_valuenum   = ('valuenum',   'mean'),
    )
    .reset_index()
)

monthly_trends['abnormal_rate'] = (
    monthly_trends['abnormal_count'] / monthly_trends['test_count'].replace(0, np.nan)
).round(4)

print(f'Monthly trends shape: {monthly_trends.shape}')
monthly_trends.head(10)

## 7. Tableau Export 4 — Admission-Type Comparison

Comparison of LOS and abnormal-test rates across admission types.

In [ ]:
admission_comparison = (
    lab.groupby('admission_type')
    .agg(
        patient_count  = ('subject_id', 'nunique'),
        total_tests    = ('itemid',     'count'),
        abnormal_tests = ('is_abnormal','sum'),
        avg_los_days   = ('los_days',   'mean'),
        median_los     = ('los_days',   'median'),
    )
    .reset_index()
)

admission_comparison['abnormal_rate'] = (
    admission_comparison['abnormal_tests'] / admission_comparison['total_tests']
).round(4)

print(f'Admission comparison shape: {admission_comparison.shape}')
admission_comparison

## 8. Export All Tableau CSVs

In [ ]:
EXPORTS = {
    'tableau_patient_summary.csv'      : patient_summary,
    'tableau_category_breakdown.csv'   : category_breakdown,
    'tableau_monthly_trends.csv'       : monthly_trends,
    'tableau_admission_comparison.csv' : admission_comparison,
}

for fname, df in EXPORTS.items():
    out_path = os.path.join(TABLEAU_DIR, fname)
    df.to_csv(out_path, index=False)
    print(f'  Saved → {os.path.relpath(out_path)}')

print('\n✅ Final load & prep complete. Tableau-ready CSVs are in data/processed/tableau_exports/')

## 9. Sign-Off

| Export File | Description | Rows |
|---|---|---|
| `tableau_patient_summary.csv` | Per-patient aggregated stats | ~100 |
| `tableau_category_breakdown.csv` | Lab results by fluid/category | ~35 |
| `tableau_monthly_trends.csv` | Monthly time-series | ~780 |
| `tableau_admission_comparison.csv` | Admission-type comparison | ~4 |

**Tableau Dashboard →** [MediScope Interactive Dashboard](https://public.tableau.com/app/profile/sambhav.kumar2781/viz/dv_p7/Dashboard1)